# 🔎 Teste Inicial de Consumo da API Pública da DEV Community

## Origem dos Dados

A DEV Community é uma comunidade com mais de três milhões de desenvolvedores de software [<a href="#ref1">1</a>]. No site, os desenvolvedores escrevem artigos (posts de blog, perguntas em fóruns de discussão, tópicos de ajuda, etc.), participam de discussões e constroem seus perfis profissionais [<a href="#ref2">2</a>].

O site é hospedado pelo FOREM [<a href="#ref2">2</a>], um software de código aberto para a construção de comunidades. Na documentação do FOREM é disponibilizada uma API pública [<a href="#ref3">3</a>] que retorna os textos publicados, tags sobre o assunto tratado, entre outras informações. A documentação da API [<a href="#ref4">4</a>] não estabelece uma licença específica para os dados disponibilizados. Os dados que podem identificar de alguma forma os autores dos textos foram ocultados.

## Objetivo

Este notebook tem como objetivo realizar um teste inicial de consumo da versão 1 [4] da API pública do DEV Community, avaliando o acesso aos endpoints disponíveis, a estrutura dos dados retornados e a viabilidade de utilização dessas informações no Hackathon ONE - Projeto TechMind para treinamento e avaliação do modelo de classificação de textos.

## Teste

In [1]:
# Importando bibliotecas
import requests
import pandas as pd
import time

from IPython.display import display
pd.reset_option('display.max_colwidth')

In [2]:
# Definindo categorias e suas tags correspondentes no DEV.to
categorias = {
    "Backend": "backend",
    "Frontend": "frontend",
    "Data Science": "datascience",
    "IA": "ai"
}

In [3]:
# Buscar artigos por categoria
artigos_dataset = []

for categoria, subcategoria in categorias.items():
    print(f"Buscando artigos de {categoria}...")

    response = requests.get(
        "https://dev.to/api/articles",
        params={
            "tag": subcategoria,
            "per_page": 5
        }
    )
    artigos = response.json()

    # Buscar o conteúdo completo (texto) de cada artigo via id
    for artigo in artigos:
        artigo_id = artigo["id"]

        response_detalhe = requests.get(
            f"https://dev.to/api/articles/{artigo_id}"
        )

        detalhe = response_detalhe.json()

        # filtrando informações do retorno
        artigos_dataset.append({
            "categoria": categoria,
            "subcategoria": subcategoria,
            "titulo": detalhe["title"],
            "texto": detalhe.get("body_markdown", ""),
            **artigo
        })

        # evitar muitas requisições seguidas
        time.sleep(0.5)
print("Dados obtidos")

# Criar DataFrame
df = pd.DataFrame(artigos_dataset)

Buscando artigos de Backend...
Buscando artigos de Frontend...
Buscando artigos de Data Science...
Buscando artigos de IA...
Dados obtidos


## Análise Exploratória

In [4]:
# Confirmando estrutura gerada e quantidades
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 32 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   categoria                 20 non-null     object 
 1   subcategoria              20 non-null     object 
 2   titulo                    20 non-null     object 
 3   texto                     20 non-null     object 
 4   type_of                   20 non-null     object 
 5   id                        20 non-null     int64  
 6   title                     20 non-null     object 
 7   description               20 non-null     object 
 8   readable_publish_date     20 non-null     object 
 9   slug                      20 non-null     object 
 10  path                      20 non-null     object 
 11  url                       20 non-null     object 
 12  comments_count            20 non-null     int64  
 13  public_reactions_count    20 non-null     int64  
 14  collection_i

Podemos verificar que:
* O retorno foi de 20 artigos. Não há dados faltantes na coluna "texto" e na coluna "tags"
* retorno da api tem algumas colunas a mais do que as descritas na documentação.
Como por exemplo a coluna "language", que talvez possa ser usada para filtrar somente textos em inglês
* A coluna flare_tag (tag de destaque) não é viável para o target porque na maior parte dos registos o valor é nulo 
* Há diversas colunas que podem possibilitar a identificação dos autores dos artigos. Todas serão ocultadas nessa análise
* A API tem uma opção de consultar excluindo textos que possuem determinada tag, o parâmetro é 'tags_exclude'. Tentamos utilizar para descartar artigos que tivessem a tag 'discuss', mas o parametro não funciona junto com o parametro 'tag', impossiblitando sua utilização

In [5]:
# Definindo colunas que serão ocultadas
# Motivo: evitar identificação do autor do texto
colunas_ocultar = ['path', 'url','canonical_url',"slug", "social_image","cover_image", 'user']

In [6]:
# Validando quantidade por categoria
df['categoria'].value_counts()

categoria
Backend         5
Frontend        5
Data Science    5
IA              5
Name: count, dtype: int64

In [7]:
# Analisando dados
df.drop(columns = colunas_ocultar).head()

,categoria,subcategoria,titulo,texto,type_of,id,title,description,readable_publish_date,comments_count,...,positive_reactions_count,created_at,edited_at,crossposted_at,published_at,last_comment_at,reading_time_minutes,tag_list,tags,flare_tag
0,Backend,backend,How Vector Databases Search a Million Vectors ...,"Take the word ""king."" Your database does not s...",article,4175403,How Vector Databases Search a Million Vectors ...,"Take the word ""king."" Your database does not s...",Jul 18,0,...,0,2026-07-18T18:53:10Z,None,None,2026-07-18T18:53:10Z,2026-07-18T18:53:10Z,4,"[ai, vectordatabase, machinelearning, backend]","ai, vectordatabase, machinelearning, backend",NaN
1,Backend,backend,"Why Redis Splits Into Exactly 16,384 Slots: A ...","---\ntitle: Why Redis Splits Into Exactly 16,3...",article,4175739,"Why Redis Splits Into Exactly 16,384 Slots: A ...","If you have ever spun up a Redis Cluster, you ...",Jul 18,1,...,1,2026-07-18T20:22:14Z,None,None,2026-07-18T20:22:13Z,2026-07-18T23:57:23Z,10,"[systemdesign, backend, redis, performance]","systemdesign, backend, redis, performance",NaN
2,Backend,backend,Synchronous vs Asynchronous Workloads: Why Mod...,## The Illusion of Simplicity\n\nMost software...,article,4173916,Synchronous vs Asynchronous Workloads: Why Mod...,The Illusion of Simplicity Most software beg...,Jul 18,0,...,0,2026-07-18T13:08:07Z,None,None,2026-07-18T16:30:00Z,2026-07-18T16:30:00Z,5,"[systemdesign, api, backend, distributedsystems]","systemdesign, api, backend, distributedsystems",NaN
3,Backend,backend,Don’t Use console.log for Logging: What Vercel...,I used AI for formatting and grammar.\n\n\nRec...,article,4173990,Don’t Use console.log for Logging: What Vercel...,I used AI for formatting and grammar. Recentl...,Jul 18,0,...,1,2026-07-18T13:22:36Z,2026-07-18T13:23:16Z,None,2026-07-18T13:22:36Z,2026-07-18T13:22:36Z,2,"[webdev, backend, ai, discuss]","webdev, backend, ai, discuss","{'name': 'discuss', 'bg_color_hex': '#71EA8B',..."
4,Backend,backend,The 837/999 HIPAA Acknowledgment Loop,\n# Every 837 You Send Needs a 999 Back: How t...,article,4174147,The 837/999 HIPAA Acknowledgment Loop,Every 837 You Send Needs a 999 Back: How the H...,Jul 18,0,...,0,2026-07-18T14:00:10Z,None,None,2026-07-18T14:00:10Z,2026-07-18T14:00:10Z,7,"[architecture, backend, monitoring, systemdesign]","architecture, backend, monitoring, systemdesign",NaN


In [ ]:
# Conferindo conteúdo por url 
# Código ocultado e output excluído para não expor os dados de identificação
'''
print(
    df.loc[
        df["id"] == 4175403,
        ["categoria", "subcategoria", "url", "social_image", "cover_image"]
    ].to_string()
)
'''

In [9]:
# Verificando language
df['language'].value_counts()

language
en    19
id     1
Name: count, dtype: int64

In [10]:
# Verificando categoria, texto e tags
df[['categoria', 'texto', 'tags']]

,categoria,texto,tags
0,Backend,"Take the word ""king."" Your database does not s...","ai, vectordatabase, machinelearning, backend"
1,Backend,"---\ntitle: Why Redis Splits Into Exactly 16,3...","systemdesign, backend, redis, performance"
2,Backend,## The Illusion of Simplicity\n\nMost software...,"systemdesign, api, backend, distributedsystems"
3,Backend,I used AI for formatting and grammar.\n\n\nRec...,"webdev, backend, ai, discuss"
4,Backend,\n# Every 837 You Send Needs a 999 Back: How t...,"architecture, backend, monitoring, systemdesign"
5,Frontend,Choosing a state management library feels a bi...,"architecture, frontend, react, typescript"
6,Frontend,> [_Artikel ini juga tersedia dalam bahasa Ing...,"webdev, frontend, nextjs, vue"
7,Frontend,"\n> Learn what a design system is, why modern ...","frontend, architecture, design, systemdesign"
8,Frontend,🚀 **WordPress 6.9 is on the way! Here's what d...,"mdrajumiah, frontend, developer, webdev"
9,Frontend,I was working on a somewhat complex client mob...,"frontend, supabase, authentication, react"


In [11]:
# Analisando um texto completo - Backend
with pd.option_context('display.max_colwidth', None):
  display(df[df['categoria'] == 'Backend'][['description', 'texto', 'tags']].iloc[[0]])

description  \
0  Take the word "king." Your database does not store the word. It stores a vector: a list of 768...   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                        

In [12]:
# Analisando um texto completo - Frontend
with pd.option_context('display.max_colwidth', None):
  display(df[df['categoria'] == 'Frontend'][['description', 'texto', 'tags']].iloc[[0]])

,description,texto,tags
5,"Choosing a state management library feels a bit like walking into a candy shop. You've got Zustand,...","Choosing a state management library feels a bit like walking into a candy shop. You've got Zustand, Redux Toolkit, Recoil, XState, and good old React Context. It’s easy to get caught up in the hype and introduce a massive library for a project that honestly doesn't need it.\n\nLet’s talk about how to audit your state management and why **""Less is More""** should be your guiding principle for your next React + TypeScript project.\n\n---\n\n### The Red Flag: ""Where did this prop come from?""\n\nWe’ve all been there. You open a component, look at the props, and see something like this:\n\n```typescript\ninterface DashboardHeaderProps {\n user: UserData;\n theme: 'light' | 'dark';\n isSidebarOpen: boolean;\n toggleSidebar: () => void;\n // ...10 more props passed down from 4 levels above\n}\nThis is Prop Drilling hell, and the natural instinct is to run straight into the arms of a global state manager. But wait—before you npm install, let's break down the types of state you actually have.\n\n1. Server State vs. Client State (The Great Divide)\n90% of the data in your frontend application usually comes from an API. If you are fetching user profiles, financial data, or product lists, that is not client state. That is Server State.\n\nIf you use a tool like TanStack Query (React Query) or framework built-ins (like Server Components), you don't need Redux or Zustand for this data.\n\n💡 The Rule: If it lives in a database and you're just fetching it, let your data-fetching library cache it. Keep it out of your global UI store.\n\n2. UI State: Local is King\nDoes the entire application need to know that a specific dropdown menu is open? No.\n\nKeep your UI state as local as possible. If two sibling components need to share a tiny piece of layout state (like a sidebar toggle), React Context combined with a clean TypeScript interface is perfectly fine.\n\nTypeScript\nexport const UIProvider = ({ children }: { children: React.ReactNode }) => {\n const [isOpen, setIsOpen] = useState(false);\n \n return (\n <UIContext.Provider value={{ isOpen, toggle: () => setIsOpen(!isOpen) }}>\n {children}\n </UIContext.Provider>\n );\n};\nWhen Do You Actually Need Global State?\nSo, when should you use something like Zustand?\n\nYou need a dedicated global state manager when you have complex, highly interactive client-side logic that crosses multiple unrelated domains. Examples include:\n\nA complex multi-step form wizard where users can go back and forth.\n\nA dynamic dashboard builder where widgets interact with each other in real-time.\n\nAn e-commerce shopping cart with complex discount rules calculated on the fly.\n\nFor these cases, a lightweight store like Zustand shines because it doesn't wrap your app in providers and plays beautifully with TypeScript inference.\n\nTypeScript\nimport { create } from 'zustand'\n\ninterface CartState {\n items: string[];\n addItem: (item: string) => void;\n}\n\nexport const useCartStore = create<CartState>((set) => ({\n items: [],\n addItem: (item) => set((state) => ({ items: [...state.items, item] })),\n}))\nSummary Checklist for Your Next Project\nBefore writing your next line of code, ask yourself:\n\nIs this data from an API? ➡️ Use TanStack Query / Server State.\n\nIs this data just for this component/page layout? ➡️ Use useState or Context.\n\nIs this dynamic, complex client logic shared globally? ➡️ Reach for Zustand.\n\nBy isolating your server state from your UI state, your code becomes modular, much easier to type with TypeScript, and significantly faster to debug.\n\nLet's Chat! 💬\nWhat’s your go-to state management stack? Are you team ""No-Global-Store"" or team Zustand? Let’s discuss in the comments below! 👇","architecture, frontend, react, typescript"


In [13]:
# Analisando um texto completo - Data Science
with pd.option_context('display.max_colwidth', None):
  display(df[df['categoria'] == 'Data Science'][['description', 'texto', 'tags']].iloc[[0]])

description  \
10  Review indoor humidity sensor coverage, high-threshold spells, and data gaps locally without turning a CSV summary into a health or building diagnosis.   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                 

In [14]:
# Analisando um texto completo -  IA
with pd.option_context('display.max_colwidth', None):
  display(df[df['categoria'] == 'IA'][['description', 'texto', 'tags']].iloc[[0]])


description  \
15  Hello, I'm Maneshwar. I'm building git-lrc, a Micro AI code reviewer that runs on every commit. It is...   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                

## Teste retorno API 

#### Validação com paramêtro (tag) inexistente - GET /api/articles/{params} 

Segundo a documentação da API, o retorno é o codigo 200, porém não informa nenhum código de erro ou o que retonaria caso não houvesse registros para a subcategoria passada como parametro (tag)

In [15]:
# Confirmando retorno da api para uma subcategoria inexistente
resposta = requests.get(
    "https://dev.to/api/articles",
    params={"tag": "tagquenaoexiste", "per_page": 1}
)
print(resposta.status_code)
print(resposta.json())

200
[]


Podemos verificar, através desse teste prático, que mesmo sem nenhum dado de retorno, a API retorna o código 200, pois a requisição em si funcionou sem exceções. Nesses casos, a API retorna um array JSON vazio (`[]`), que o Python converte em uma lista vazia após o `.json()`.

Sendo assim, se faz necessário tratar o retorno no código da coleta de dados por subcategoria, visto que há possibilidade de consultar uma inexistente na base da Dev Community. 

#### Validação com paramêtro (id) inexistente - GET /api/articles/{id} 

Para o endpoint de busca de artigo por id (`/articles/{id}`), a documentação informa dois retornos possíveis: 200, para um id válido, e 404, para um id inexistente.

In [16]:
# Teste 1: id válido (deve retornar 200)
resposta_valida = requests.get("https://dev.to/api/articles/4175403")
print(resposta_valida.status_code)
print(resposta_valida.json().get('title'))

200
How Vector Databases Search a Million Vectors Without Checking a Million


Ao consultar um id válido (4175403), a API retornou o código 200 com os dados completos do artigo correspondente.

In [17]:
# Teste 2: id inexistente (deve retornar 404, segundo a documentação)
resposta_invalida = requests.get("https://dev.to/api/articles/-1")
print(resposta_invalida.status_code)
print(resposta_invalida.json())

404
{'error': 'not found', 'status': 404}


Ao consultar um id inexistente (-1), a API retornou o código 404, junto com um corpo JSON informando o erro.

Diferente do endpoint de listagem por tag, este endpoint retorna um erro HTTP real e documentado quando o artigo não é localizado. Essa informação indica que existe a possibilidade de consultar o detalhe de um artigo cujo ID já não é mais válido, por exemplo, quando o artigo foi excluído no intervalo entre as duas chamadas (listagem e detalhe).

## Primeiros Insights


* A coluna descrição não é um resumo. Seria somente um trecho do texto completo

* Conforme documentação da API e análise superficial dos textos, pode haver post de blog com conteúdo técnico mas também pode haver outros tipos de texto como comentários sobre um post ou somente um embed (um link que foi compartilhado em alguma postagem), por exemplo.  

  Isso sugere a necessidade de definir alguns parâmetros de limpeza dos dados, na fase de ETL, como excluir links e/ou manter somente registros com textos acima de uma determinada quantidade de caracteres.

* Os textos tem mais de uma lingua, necessário filtrar somente inglês para treino

* Aparentemente as tags representam os assuntos do texto.

* Há textos com emoji

## Conclusão e Recomendações

* Não houve problemas no acesso da API. Dados retornados conforme informados na documentação

* Até o momento não foi localizada informação clara sobre a licença de utilização dos dados

* É necessário uma avaliação mais profunda dos dados

* A proposta do projeto é simular um cenário real de mercado, no qual os dados chegam com inconsistências e problemas de qualidade. Por isso, a coleta será feita de forma bruta, sem filtros ou validações e sem tratamento específico para registros inexistentes ou vazios. A única exceção é o filtro de idioma, considerando apenas artigos em inglês. Caso a base seja utilizada no MVP, os ajustes serão realizados posteriormente na etapa de ETL.

* Recomendações: 
    * Obter uma quantidade maior de dados
    * Inicialmente coleta de dados brutos para posteriormente realizar ETL
    * Separar amostras por categoria para avaliar a qualidade dos dados, visando o objetivo do projeto
    * Tratamento básico de erros no consumo da API

## Referências



* <a id="ref1"></a>**[1]** DEV COMMUNITY. *Site*. Disponível em: <https://dev.to/>. Acesso em: 9 jul. 2026.

* <a id="ref2"></a>**[2]** FOREM. *Sobre dev.to*. Disponível em: <https://github.com/forem/forem/>. Acesso em: 9 jul. 2026.

* <a id="ref3"></a>**[3]** API. *Versões da API*. Disponível em: <https://developers.forem.com/api>. Acesso em: 9 jul. 2026.

* <a id="ref4"></a>**[4]** Forem API V1. *Documentação da API*. Disponível em: <https://developers.forem.com/api/v1>. Acesso em: 9 jul. 2026.